# Brent Crude Oil Analysis

I started this project to learn how to work with real financial market data in Python in the context of the price of oil during the Iran conflict. The global relevance of the stock price and large amount of information available made it an appealing setting for my first quantitative project. NOTE: cell outputs are not included in this file due to their large size, to find them look in 'Brent_Crude_Cell_Outputs' and follow along using the headings.

**AI use:** I used AI extensively while writing and debugging parts of the Python in this notebook. It was used as a coding resource when I got stuck, however I chose the questions to investigate and checked the code and results myself.

## 1. Get and clean the data

I downloaded one-minute Brent data from HistData for 2023 to August 2026. There are more than 1.1 million observations once the files are combined.

In [ ]:
%pip install histdata pandas scipy numpy

In [ ]:
from pathlib import Path
from histdata import download_hist_data
from histdata.api import Platform, TimeFrame

Path("brent_data").mkdir(exist_ok=True)

# Download full years first
for year in ["2023", "2024", "2025"]:
    print("Downloading", year)
    download_hist_data(
        year=year,
        month=None,
        pair="bcousd",
        platform=Platform.GENERIC_ASCII,
        time_frame=TimeFrame.ONE_MINUTE,
        output_directory="brent_data"
    )

# 2026 is downloaded month by month because the data is only partial
for month in range(1, 9):
    print("Downloading 2026 month", month)
    download_hist_data(
        year="2026",
        month=str(month),
        pair="bcousd",
        platform=Platform.GENERIC_ASCII,
        time_frame=TimeFrame.ONE_MINUTE,
        output_directory="brent_data"
    )

In [ ]:
import pandas as pd
import zipfile
from pathlib import Path
import re

all_data = []

for zip_path in sorted(Path("brent_data").glob("*.zip")):
    with zipfile.ZipFile(zip_path) as z:
        csv_files = [name for name in z.namelist() if name.lower().endswith(".csv")]

        for filename in csv_files:
            df = pd.read_csv(
                z.open(filename),
                sep=";",
                header=None,
                names=["timestamp", "open", "high", "low", "close", "volume"]
            )
            df["source_file"] = zip_path.name
            all_data.append(df)

brent_all = pd.concat(all_data, ignore_index=True)
print(f"Raw observations: {len(brent_all):,}")

In [ ]:
# Turn the timestamps into real datetimes and remove bad ones
brent_all["timestamp"] = pd.to_datetime(
    brent_all["timestamp"],
    format="%Y%m%d %H%M%S",
    errors="coerce"
)

brent_all = brent_all.dropna(subset=["timestamp"])
brent_all = brent_all.sort_values("timestamp").reset_index(drop=True)

# Check duplicate timestamps before cleaning them
duplicate_mask = brent_all["timestamp"].duplicated(keep=False)
print("Rows with duplicate timestamps:", duplicate_mask.sum())

# Where there are duplicates, keep the non-flat OHLC observation if there is one.
duplicate_rows = brent_all[duplicate_mask].copy()
duplicate_rows["is_flat"] = (
    (duplicate_rows["open"] == duplicate_rows["high"]) &
    (duplicate_rows["high"] == duplicate_rows["low"]) &
    (duplicate_rows["low"] == duplicate_rows["close"])
)

unique_rows = brent_all[~duplicate_mask].copy()
resolved_duplicates = duplicate_rows[~duplicate_rows["is_flat"]].copy()

brent_all = pd.concat([unique_rows, resolved_duplicates], ignore_index=True)

# Make sure there is only one row per timestamp
brent_all = (
    brent_all
    .sort_values("timestamp")
    .drop_duplicates("timestamp", keep="first")
    .reset_index(drop=True)
)

print(f"Final observations: {len(brent_all):,}")
print("Missing timestamps:", brent_all["timestamp"].isna().sum())
print("Duplicate timestamps:", brent_all["timestamp"].duplicated().sum())
print("Chronologically sorted:", brent_all["timestamp"].is_monotonic_increasing)
print("First observation:", brent_all["timestamp"].min())
print("Last observation:", brent_all["timestamp"].max())

## 2. A quick look at the data

Before testing anything specific, I looked at minute-by-minute returns and the biggest moves in the dataset.

In [ ]:
import numpy as np

brent_analysis = brent_all.copy()
brent_analysis["log_return"] = np.log(
    brent_analysis["close"] / brent_analysis["close"].shift(1)
)

largest_moves = (
    brent_analysis.loc[
        brent_analysis["log_return"].abs().nlargest(30).index,
        ["timestamp", "open", "high", "low", "close", "log_return", "source_file"]
    ]
    .sort_values("timestamp")
)

display(largest_moves)
print("\nReturn summary:")
display(brent_analysis["log_return"].describe())

## 3. Trump events and Brent prices

I then looked at 14 scheduled Trump events where Iran was expected to be a topic or was discussed. I compared Brent's movement before the scheduled start with its movement after the actual start.



In [ ]:
from scipy import stats

# Times are recorded in US Eastern Time in this list.
events_raw = [
    {"date":"2026-03-07", "scheduled_et":"10:00", "actual_et":"09:33", "event":"Shield of the Americas Summit"},
    {"date":"2026-03-09", "scheduled_et":"16:30", "actual_et":"16:50", "event":"House Republican Conference"},
    {"date":"2026-03-09", "scheduled_et":"17:30", "actual_et":"17:52", "event":"Trump news conference at Doral"},
    {"date":"2026-03-17", "scheduled_et":"14:00", "actual_et":"14:00", "event":"Ireland bilateral / Oval Office remarks"},
    {"date":"2026-03-19", "scheduled_et":"11:30", "actual_et":"11:46", "event":"Japan bilateral / Oval Office"},
    {"date":"2026-03-20", "scheduled_et":"10:30", "actual_et":"10:52", "event":"Navy Midshipmen trophy presentation"},
    {"date":"2026-03-23", "scheduled_et":"11:30", "actual_et":"11:39", "event":"Memphis Safe Task Force event"},
    {"date":"2026-03-25", "scheduled_et":"19:30", "actual_et":"19:49", "event":"NRCC dinner"},
    {"date":"2026-03-27", "scheduled_et":"12:30", "actual_et":"12:41", "event":"American farmers event"},
    {"date":"2026-03-27", "scheduled_et":"16:30", "actual_et":"16:40", "event":"Miami event"},
    {"date":"2026-03-31", "scheduled_et":"17:30", "actual_et":"17:41", "event":"Oval Office signing / reporter questions"},
    {"date":"2026-04-01", "scheduled_et":"21:00", "actual_et":"21:00", "event":"Address to the Nation"},
    {"date":"2026-04-06", "scheduled_et":"10:45", "actual_et":"10:48", "event":"White House Easter Egg Roll"},
    {"date":"2026-04-06", "scheduled_et":"13:00", "actual_et":"13:08", "event":"President's news conference"},
]

events = pd.DataFrame(events_raw)
events["scheduled_et"] = pd.to_datetime(events["date"] + " " + events["scheduled_et"])
events["actual_et"] = pd.to_datetime(events["date"] + " " + events["actual_et"])

# HistData timestamps were treated as UTC-5 in my original analysis.
events["scheduled_time"] = events["scheduled_et"] - pd.Timedelta(hours=1)
events["actual_start_time"] = events["actual_et"] - pd.Timedelta(hours=1)

In [ ]:
price_series = (
    brent_all.set_index("timestamp")["close"]
    .sort_index()
)

pre_windows = [5, 15, 30, 60, 120]
post_windows = [5, 15, 30, 60, 120]

def price_before(t):
    values = price_series.loc[price_series.index <= t]
    return values.iloc[-1] if len(values) else np.nan

def price_after(t):
    values = price_series.loc[price_series.index >= t]
    return values.iloc[0] if len(values) else np.nan

rows = []

for _, event in events.iterrows():
    scheduled = event["scheduled_time"]
    actual = event["actual_start_time"]

    p_scheduled = price_after(scheduled)
    p_actual = price_after(actual)

    row = {
        "date": event["date"],
        "event": event["event"],
        "scheduled_time": scheduled,
        "actual_start_time": actual,
        "delay_minutes": (actual - scheduled).total_seconds() / 60,
        "waiting_return": np.log(p_actual / p_scheduled) if p_scheduled > 0 and p_actual > 0 else np.nan,
    }

    for minutes in pre_windows:
        p_pre = price_before(scheduled - pd.Timedelta(minutes=minutes))
        row[f"pre_{minutes}m"] = np.log(p_scheduled / p_pre) if p_pre > 0 and p_scheduled > 0 else np.nan

    for minutes in post_windows:
        p_post = price_after(actual + pd.Timedelta(minutes=minutes))
        row[f"post_{minutes}m"] = np.log(p_post / p_actual) if p_actual > 0 and p_post > 0 else np.nan

    rows.append(row)

event_returns = pd.DataFrame(rows)
display(event_returns)

### Testing whether the pre-event movement was related to the post-event movement

My main question here was whether a price move just before an event was related to the move afterwards. I tested Pearson and Spearman correlations and also checked whether the direction was the same more often than 50%.

In [ ]:
results = []

for pre in pre_windows:
    for post in post_windows:
        temp = event_returns[[f"pre_{pre}m", f"post_{post}m"]].dropna()
        if len(temp) < 4:
            continue

        x = temp[f"pre_{pre}m"]
        y = temp[f"post_{post}m"]

        r, p = stats.pearsonr(x, y)
        rho, rho_p = stats.spearmanr(x, y)
        same_direction = np.sign(x) == np.sign(y)
        direction_test = stats.binomtest(same_direction.sum(), n=len(temp), p=0.5)

        results.append({
            "pre_minutes": pre,
            "post_minutes": post,
            "N": len(temp),
            "pearson_r": r,
            "pearson_p": p,
            "spearman_rho": rho,
            "spearman_p": rho_p,
            "directional_accuracy": same_direction.mean(),
            "directional_p": direction_test.pvalue,
        })

robustness = pd.DataFrame(results)
robustness["bonferroni_p"] = (robustness["pearson_p"] * len(robustness)).clip(upper=1)
robustness = robustness.sort_values("pearson_p")

display(robustness)

### A simple robustness check

I also repeated the directional test while leaving one event out at a time. This was useful for seeing how much the result depended on one particular event.

In [ ]:
sign_data = event_returns.copy()

for minutes in pre_windows:
    sign_data[f"pre_sign_{minutes}m"] = np.sign(sign_data[f"pre_{minutes}m"])
    sign_data[f"post_sign_{minutes}m"] = np.sign(sign_data[f"post_{minutes}m"])

loo_results = []

for minutes in pre_windows:
    pre_col = f"pre_sign_{minutes}m"
    post_col = f"post_sign_{minutes}m"
    base = sign_data[[pre_col, post_col]].dropna()
    base = base[(base[pre_col] != 0) & (base[post_col] != 0)]

    for i in range(len(base)):
        temp = base.drop(base.index[i])
        if len(temp) == 0:
            continue
        correct = (temp[pre_col] == temp[post_col]).sum()
        test = stats.binomtest(correct, n=len(temp), p=0.5)
        loo_results.append({
            "pre_window_minutes": minutes,
            "removed_row": i,
            "accuracy": correct / len(temp),
            "p_value": test.pvalue,
        })

loo_directional = pd.DataFrame(loo_results)
display(loo_directional[loo_directional["pre_window_minutes"] == 60])

## 4. Holding Brent over the weekend during the Iran conflict

I also wanted to test a much simpler idea: during the conflict period, was it better to stay in a position over the weekend or close it on Friday and reopen on Monday?

I only had a small number of complete weeks, so this was mainly a learning exercise.

In [ ]:
conflict_start = pd.Timestamp("2026-02-28")
one_way_cost = 0.0005
round_trip_cost = 2 * one_way_cost

df = brent_all[brent_all["timestamp"] >= conflict_start].copy()
df["weekday"] = df["timestamp"].dt.dayofweek

df = df[df["weekday"].between(0, 4)].copy()
df["log_return"] = np.log(df["close"] / df["close"].shift(1))

df["week_start_date"] = (
    df["timestamp"] - pd.to_timedelta(df["weekday"], unit="D")
).dt.normalize()

weekly = (
    df.groupby("week_start_date")
      .agg(
          week_start=("timestamp", "min"),
          week_end=("timestamp", "max"),
          monday_open=("open", "first"),
          friday_close=("close", "last"),
          intraday_volatility=("log_return", "std"),
      )
      .reset_index()
)

weekly = weekly[
    (weekly["week_start"].dt.dayofweek == 0) &
    (weekly["week_end"].dt.dayofweek == 4) &
    (weekly["week_start"] >= pd.Timestamp("2026-03-02"))
].copy()

weekly["realised_volatility"] = weekly["intraday_volatility"] * np.sqrt(252 * 24 * 60)

In [ ]:
# Find the last Brent price on each Friday and first price on the following Monday
fridays = (
    df[df["weekday"] == 4]
    .sort_values("timestamp")
    .groupby(df.loc[df["weekday"] == 4, "timestamp"].dt.normalize())
    .tail(1)[["timestamp", "close"]]
    .rename(columns={"timestamp":"friday_time", "close":"friday_close"})
)
fridays["friday_date"] = fridays["friday_time"].dt.normalize()

mondays = (
    df[df["weekday"] == 0]
    .sort_values("timestamp")
    .groupby(df.loc[df["weekday"] == 0, "timestamp"].dt.normalize())
    .head(1)[["timestamp", "open"]]
    .rename(columns={"timestamp":"monday_time", "open":"monday_open"})
)
mondays["monday_date"] = mondays["monday_time"].dt.normalize()

fridays["following_monday"] = fridays["friday_date"] + pd.Timedelta(days=3)
weekend = fridays.merge(mondays, left_on="following_monday", right_on="monday_date", how="inner")

weekend["weekend_return_long"] = weekend["monday_open"] / weekend["friday_close"] - 1
weekend["weekend_return_short"] = -weekend["weekend_return_long"]
weekend["hold_advantage_long"] = weekend["weekend_return_long"] + round_trip_cost
weekend["hold_advantage_short"] = weekend["weekend_return_short"] + round_trip_cost

weekly["friday_date"] = weekly["week_end"].dt.normalize()

weekly = weekly.merge(
    weekend[["friday_date", "weekend_return_long", "weekend_return_short", "hold_advantage_long", "hold_advantage_short"]],
    on="friday_date",
    how="inner"
)

print("Complete weekends:", len(weekly))
display(weekly)

### Does weekday volatility give any clue about the weekend result?

I compared realised weekday volatility with the advantage of holding over the weekend. I used correlation tests and a leave-one-week-out prediction test.

In [ ]:
def analyse_weekend(data, advantage_col):
    x = data["realised_volatility"].astype(float)
    y = data[advantage_col].astype(float)

    pearson_r, pearson_p = stats.pearsonr(x, y)
    spearman_rho, spearman_p = stats.spearmanr(x, y)
    regression = stats.linregress(x, y)

    return {
        "N": len(data),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "r_squared": regression.rvalue ** 2,
        "regression_p": regression.pvalue,
        "hold_win_rate": (y > 0).mean(),
    }

weekend_test = weekly.dropna().reset_index(drop=True)

weekend_results = pd.DataFrame([
    analyse_weekend(weekend_test, "hold_advantage_long") | {"direction":"Long"},
    analyse_weekend(weekend_test, "hold_advantage_short") | {"direction":"Short"},
])

display(weekend_results)

In [ ]:
def leave_one_out_accuracy(data, advantage_col):
    results = []

    for i in range(len(data)):
        train = data.drop(index=i)
        test_row = data.iloc[i]

        model = stats.linregress(
            train["realised_volatility"],
            train[advantage_col]
        )

        predicted = model.intercept + model.slope * test_row["realised_volatility"]
        predicted_strategy = "HOLD" if predicted > 0 else "CLOSE"
        actual_strategy = "HOLD" if test_row[advantage_col] > 0 else "CLOSE"

        results.append(predicted_strategy == actual_strategy)

    accuracy = np.mean(results)
    p_value = stats.binomtest(sum(results), n=len(results), p=0.5).pvalue
    return accuracy, p_value

long_accuracy, long_p = leave_one_out_accuracy(weekend_test, "hold_advantage_long")
short_accuracy, short_p = leave_one_out_accuracy(weekend_test, "hold_advantage_short")

print(f"Long leave-one-week-out accuracy:  {long_accuracy:.1%}, p = {long_p:.4f}")
print(f"Short leave-one-week-out accuracy: {short_accuracy:.1%}, p = {short_p:.4f}")

## 5. What I learned

The main thing I got from this project was experience working with a large real dataset and turning questions about market behaviour into simple tests. I also learned that it can appear very easy to find patterns in market data however that does not mean they hold up once they are tested properly.

This project was key in developing my Python skills. I had to work through data cleaning, time-series handling, statistical tests and debugging, and I became much more comfortable using AI and other resources to get through unfamiliar coding problems.